# Full-information secretary problem: $n=2$ and $n=3$

Observe $n$ independent $U(0,1)$ values sequentially. The objective is to stop on the **overall maximum**. Unlike the classical relative-rank problem, we observe each numerical value.

This notebook analyzes the **first decision only** for $n=2$ and $n=3$ with an exact continuation formula, and then simulates the optimal policy conditional on a chosen first observation $x$. It does not claim to solve arbitrary horizons.


## Mathematical stopping decision after observing $x$

Accepting the first observation wins with probability

$$
S_n(x)=x^{n-1}.
$$

Rejecting it and then acting optimally gives

$$
C_2(x)=1-x,
\qquad
C_3(x)=
\begin{cases}
\frac34-\frac{x^2}{2},&0\le x\le\frac12,\\
\frac12+x-\frac{3x^2}{2},&\frac12<x\le1.
\end{cases}
$$

For $n=3$, after rejecting $x$ the second observation $y$ is accepted only when $y>x$ and $y\ge1/2$; otherwise the last observation is chosen. The optimal first-stage cutoffs are $1/2$ for $n=2$ and $(1+\sqrt6)/5$ for $n=3$.

At $x$ choose the action with the higher conditional probability.


In [ ]:
from fractions import Fraction
from math import sqrt
import random

def stop_value(n,x):
    if n not in (2,3) or not 0 <= x <= 1:
        raise ValueError("n must be 2 or 3; x must be in [0,1].")
    return x**(n-1)

def continue_value(n,x):
    if n not in (2,3) or not 0 <= x <= 1:
        raise ValueError("n must be 2 or 3; x must be in [0,1].")
    if n==2:
        return 1-x
    if x <= Fraction(1,2):
        return Fraction(3,4) - x*x/2
    return Fraction(1,2)+x-Fraction(3,2)*x*x

def cutoff(n):
    return .5 if n==2 else (1+sqrt(6))/5

n, x = 3, Fraction("0.70")
exact_stop, exact_continue = stop_value(n,x), continue_value(n,x)
print(f"First observation x={float(x):.2f}, n={n}")
print(f"Accept now: {float(exact_stop):.4%}")
print(f"Continue:   {float(exact_continue):.4%}")
print("Optimal action:", "ACCEPT" if exact_stop >= exact_continue else "CONTINUE")
print(f"First-stage cutoff: {cutoff(n):.6f}")
assert continue_value(3,Fraction(1,2))==Fraction(5,8)


## Conditional Monte Carlo experiment

We **fix the first observation** $x$ and independently simulate the remaining value(s). On each simulated continuation, we compare what would happen if the first value were selected immediately versus rejected.

For $n=3$, the second-stage decision can use $y$ and the fact that $x$ was rejected, but not the unknown final value $z$. In the last period a nonrecord cannot win; accepting the last value regardless of record status gives the same winning probability as having no selection in that case.


In [ ]:
def one_trial(n, x, rng):
    future = [rng.random() for _ in range(n-1)]
    stop_wins = all(y < float(x) for y in future)
    if n == 2:
        continue_wins = future[0] > float(x)
    else:
        y,z = future
        if y > float(x) and y >= 0.5:
            continue_wins = y > z
        else:
            continue_wins = z > max(float(x),y)
    return stop_wins, continue_wins

def simulate_conditional(n, x, trials=20000, seed=2026):
    if trials <= 0:
        raise ValueError("trials must be positive.")
    stop_value(n,x) # validate
    rng = random.Random(seed)
    wins_stop = wins_continue = 0
    for _ in range(trials):
        stop, cont = one_trial(n,x,rng)
        wins_stop += stop
        wins_continue += cont
    return wins_stop/trials, wins_continue/trials

estimated_stop, estimated_continue = simulate_conditional(n,x)
print(f"Accept now: exact {float(exact_stop):.3%}, simulation {estimated_stop:.3%}")
print(f"Continue:   exact {float(exact_continue):.3%}, simulation {estimated_continue:.3%}")


## Explore the decision boundary

The two curves show the conditional probability of eventually selecting the overall maximum, after observing the first value $x$. The crossover is the first-stage optimal cutoff.


In [ ]:
xs = [Fraction(i,100) for i in range(101)]
stop_curve = [float(stop_value(n,t)) for t in xs]
continue_curve = [float(continue_value(n,t)) for t in xs]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.plot([float(t) for t in xs],stop_curve,label="Accept first observation")
    plt.plot([float(t) for t in xs],continue_curve,label="Reject; then continue optimally")
    plt.axvline(cutoff(n),linestyle="--",label=f"Cutoff {cutoff(n):.4f}")
    plt.xlabel("First observation x")
    plt.ylabel("Conditional probability of success")
    plt.legend()
    plt.show()
except ImportError:
    print("Install matplotlib to view the optional figure.")


## Exercises

1. Change to $n=2$. Verify that the cutoff is $x=1/2$.
2. Set $n=3$ and test $x=0.60$ and $x=0.75$. What action is optimal?
3. Derive $C_3(x)$ by integrating the second-stage rule over $y$.
4. Explain why the observations must be compared with the running maximum, not simply with the first value.

**Reference:** Gilbert and Mosteller, *Recognizing the Maximum of a Sequence* (1966).
